# RAG Pipeline Walkthrough — Week 2

### GenAI Roadmap — Week 2 Mini-Project

The chatbot in [`app.py`](./app.py) hides its pipeline behind a chat box. This notebook
opens it up: every stage from **PDF → chunks → embeddings → ChromaDB → retrieval →
generation** is run in isolation so you can see what each one produces and what changes
when you tune it.

Stack: **LangChain** for the abstractions, **ChromaDB** for the vector store,
**Groq** for generation (standing in for OpenAI), **fastembed** for local embeddings.

The code lives in [`src/`](./src) and [`prompts/`](./prompts) — the notebook only calls it,
the same as the app does. See [README.md](./README.md) for the layout.

**Before running:** put at least one PDF in `data/pdfs/`, or run `python ingest.py <file.pdf>`.

## 0. Setup

In [ ]:
import json
from pathlib import Path

from src import config
from src.embeddings import get_embeddings
from src.ingest import ingest_paths
from src.llm import get_llm
from src.loaders import load_pdfs, split_documents
from src.vectorstore import clear_collection, count, get_vectorstore, list_sources

PDFS = sorted(config.PDF_DIR.glob("*.pdf"))
print(f"PDFs found in {config.PDF_DIR}: {[p.name for p in PDFS] or 'none — add some before continuing'}")
print(f"Embedding model: {config.EMBEDDING_MODEL} ({config.EMBEDDING_BACKEND})")
print(f"LLM: {config.GROQ_MODEL}")

## 1. Document loading

One `Document` per page. The metadata is what makes citation possible later — an answer
that says "page 7 of handbook.pdf" is checkable, one that just asserts a fact is not.

In [ ]:
pages = load_pdfs(config.PDF_DIR)
print(f"{len(pages)} pages loaded from {len(PDFS)} file(s)\n")

sample = pages[0]
print("Metadata:", {k: sample.metadata[k] for k in ("source", "page") if k in sample.metadata})
print("\nFirst 400 characters:\n")
print(sample.page_content[:400])

## 2. Chunking

A page is the wrong retrieval unit in both directions: too big to embed precisely (one
vector has to represent several unrelated paragraphs) and too big to inject cheaply.
Chunking trades that off.

Two parameters do the work:

- **`chunk_size`** — small chunks give sharp, focused embeddings but can cut an
  explanation in half; large chunks keep context but blur the embedding across topics.
- **`chunk_overlap`** — repeats the tail of each chunk at the head of the next, so a fact
  spanning a boundary still appears intact in at least one chunk.

Below, the same pages are split three ways. Watch the chunk count and the average length
move together — that is the cost side of the trade.

In [ ]:
for size, overlap in [(500, 50), (1000, 150), (2000, 200)]:
    chunks = split_documents(pages, chunk_size=size, chunk_overlap=overlap)
    lengths = [len(c.page_content) for c in chunks]
    print(
        f"size={size:>4} overlap={overlap:>3} -> {len(chunks):>4} chunks, "
        f"avg {sum(lengths) // len(lengths):>4} chars, max {max(lengths):>4}"
    )

In [ ]:
chunks = split_documents(pages)  # the configured defaults
print(f"Using size={config.CHUNK_SIZE} overlap={config.CHUNK_OVERLAP} -> {len(chunks)} chunks\n")
print(chunks[1].page_content[:300], "...\n")
print("Metadata:", chunks[1].metadata)

### Seeing the overlap

The end of one chunk and the start of the next share text. That duplication is the point:
it is insurance against a boundary landing in the middle of the sentence that answers
someone's question.

In [ ]:
a, b = chunks[1].page_content, chunks[2].page_content
tail = a[-config.CHUNK_OVERLAP:]
print(f"Tail of chunk 1:\n...{tail}\n")
print(f"Head of chunk 2:\n{b[:config.CHUNK_OVERLAP]}...\n")
print("Overlap present:", b.startswith(tail[:40]) or tail[-40:] in b[:400])

## 3. Embeddings

An embedding maps text to a fixed-length vector positioned so that semantically similar
text lands nearby. That is the whole trick behind retrieval: instead of matching words,
you measure distance.

`BAAI/bge-small-en-v1.5` runs locally through fastembed — no API key, no torch, and it
works on a free Hugging Face Space.

In [ ]:
embeddings = get_embeddings()
vector = embeddings.embed_query("What does this document say about pricing?")
print(f"Dimensions: {len(vector)}")
print(f"First 8 values: {[round(v, 4) for v in vector[:8]]}")

### Why it beats keyword matching

Cosine similarity between embeddings of sentences that share no vocabulary. The pairs
that mean the same thing score high anyway — that is what a keyword index cannot do.

In [ ]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm = (sum(x * x for x in a) ** 0.5) * (sum(y * y for y in b) ** 0.5)
    return dot / norm


sentences = [
    "How much annual leave am I entitled to?",
    "What is the company's vacation day policy?",
    "The server returned a 500 internal error.",
]
vectors = embeddings.embed_documents(sentences)

print(f"same meaning, no shared words : {cosine(vectors[0], vectors[1]):.3f}")
print(f"unrelated topics             : {cosine(vectors[0], vectors[2]):.3f}")

## 4. Vector store — ChromaDB

Chroma persists to disk, so an index built once survives restarts. Chunk ids are content
hashes, which makes ingestion idempotent: re-running it on an unchanged file upserts the
same rows instead of piling up duplicates.

`clear_collection` empties the collection through the open client rather than deleting the
directory. On Windows the client holds its SQLite file open, so removing the folder
underneath it half-succeeds and leaves a store that fails on the next write.

In [ ]:
vectorstore = get_vectorstore()
clear_collection(vectorstore)  # start clean so the counts below are unambiguous
summary = ingest_paths(PDFS, vectorstore)

for entry in summary:
    print(f"  {entry['source']}: {entry['pages']} pages -> {entry['chunks']} chunks")
print(f"\nCollection: {count(vectorstore)} chunks from {list_sources(vectorstore)}")

In [ ]:
# Idempotence check: same files again, same total.
ingest_paths(PDFS, vectorstore)
print(f"After re-ingesting the same files: {count(vectorstore)} chunks")

## 5. Retrieval strategies

Three ways to pick which chunks reach the prompt. Ask a question your documents actually
cover and compare what comes back.

In [ ]:
QUERY = "What is this document about?"  # replace with something specific to your PDFs

In [ ]:
from src.retrieval import build_bm25, fetch_all_documents, hybrid_search, mmr_search, similarity_search


def preview(documents, label):
    print(f"--- {label} ---")
    for i, doc in enumerate(documents, start=1):
        text = " ".join(doc.page_content.split())[:110]
        print(f"  [{i}] p{doc.metadata.get('page', '?')} {doc.metadata.get('source', '')}: {text}…")
    print()


preview(similarity_search(vectorstore, QUERY, k=4), "similarity — nearest neighbours")
preview(mmr_search(vectorstore, QUERY, k=4), "MMR — relevance minus redundancy")
preview(hybrid_search(vectorstore, QUERY, k=4), "hybrid — dense + BM25, fused by RRF")

### Where each one wins

- **similarity** is the default and usually fine. Its failure mode is redundancy: if a
  passage is repeated across the corpus, the top *k* can be *k* copies of one fact.
- **MMR** fixes exactly that, at the cost of some relevance. Worth it for broad questions
  that need several parts of a document.
- **hybrid** covers the case embeddings are worst at — rare literal tokens. An error code,
  a part number, an unusual name all embed to something generic, and BM25 finds them exactly.

The query below is deliberately keyword-shaped, where the gap tends to show.

In [ ]:
KEYWORD_QUERY = "section 4.2"  # try an identifier, code, or rare term from your documents

bm25 = build_bm25(fetch_all_documents(vectorstore))
preview(similarity_search(vectorstore, KEYWORD_QUERY, k=3), "dense only")
preview(bm25.invoke(KEYWORD_QUERY)[:3], "BM25 only")
preview(hybrid_search(vectorstore, KEYWORD_QUERY, k=3, bm25_retriever=bm25), "hybrid")

### Reciprocal rank fusion

Hybrid search cannot just add a cosine similarity to a BM25 score — the two are not on a
comparable scale and normalizing them is guesswork. Rank positions *are* comparable, so
RRF scores each document by `1 / (60 + rank)` summed across the lists it appears in. A
document ranked well by both retrievers beats one ranked first by only one of them.

In [ ]:
from src.retrieval import reciprocal_rank_fusion

dense = similarity_search(vectorstore, KEYWORD_QUERY, k=10)
sparse = bm25.invoke(KEYWORD_QUERY)[:10]
fused = reciprocal_rank_fusion([dense, sparse])

ids = lambda docs: [d.metadata.get("chunk_id", "")[:8] for d in docs]
print(f"dense  top-5: {ids(dense)[:5]}")
print(f"sparse top-5: {ids(sparse)[:5]}")
print(f"fused  top-5: {ids(fused)[:5]}")

## 6. Re-ranking

Retrieval wants recall — cast wide so the right chunk is somewhere in the candidate set.
Generation wants precision, because irrelevant context measurably degrades the answer.

A re-ranker bridges them. The key difference from the embedding: it reads the query and
the candidate **together**, whereas the embedding encoded each one separately with no
knowledge of the other. Production systems use a cross-encoder; this one scores with the
LLM, which costs an extra call but keeps the project on a single provider.

In [ ]:
from src.retrieval import rerank

llm = get_llm()
candidates = similarity_search(vectorstore, QUERY, k=10)
reranked = rerank(llm, QUERY, candidates, top_n=4)

preview(candidates[:4], "top 4 before re-ranking")
preview(reranked, "top 4 after re-ranking")

## 7. Context injection

Retrieved chunks have to be *placed* in the prompt, and how you place them matters as
much as which ones you picked. Two decisions in [`prompts/answer.py`](./prompts/answer.py):

1. Each chunk is numbered and labelled with its filename and page, giving the model a
   stable handle to cite. Ask for citations without providing identifiers and it invents them.
2. The grounding rules come *before* the context, so they are not buried under whatever
   the retriever returned.

Here is the exact text the model receives.

In [ ]:
from prompts.answer import ANSWER_PROMPT, format_context

context = format_context(similarity_search(vectorstore, QUERY, k=2))
messages = ANSWER_PROMPT.invoke({"context": context, "chat_history": [], "question": QUERY}).messages

for message in messages:
    print(f"===== {message.type.upper()} =====")
    print(message.content[:1500])
    print()

## 8. Conversation memory

This is the part that makes it a *chatbot* rather than a search box.

The retriever embeds the question and nothing else. So a follow-up like *"why is that?"*
embeds to noise — the vector carries no information about what "that" was. Conversation
history in the answer prompt does not help, because retrieval already happened by then.

The fix is **query transformation**: rewrite the follow-up against the history into a
standalone question *before* retrieving. Below, the same follow-up is retrieved for with
and without the rewrite.

In [ ]:
from src.chain import build_condense_chain
from src.history import to_messages

condense = build_condense_chain(llm)

history = to_messages([
    {"role": "user", "content": "What are the main topics covered in these documents?"},
    {"role": "assistant", "content": "They cover the retrieval pipeline, chunking strategy, and evaluation."},
])
FOLLOW_UP = "Can you say more about the second one?"

rewritten = condense.invoke({"question": FOLLOW_UP, "chat_history": history})
print(f"Raw follow-up : {FOLLOW_UP}")
print(f"Rewritten     : {rewritten}\n")

preview(similarity_search(vectorstore, FOLLOW_UP, k=3), "retrieved for the RAW follow-up")
preview(similarity_search(vectorstore, rewritten, k=3), "retrieved for the REWRITTEN question")

## 9. The full pipeline

Everything above, composed. `RAGChatbot` is what `app.py` drives; the LCEL chain
underneath it is in [`src/chain.py`](./src/chain.py).

In [ ]:
from src.chain import RAGChatbot

bot = RAGChatbot(vectorstore)
conversation = []

for question in [
    "What is the main subject of these documents?",
    "What does it say about that in more detail?",  # deliberately dependent on the answer above
]:
    result = bot.answer(question, to_messages(conversation))
    print(f"Q: {question}")
    if result["standalone_question"] != question:
        print(f"   (rewritten to: {result['standalone_question']})")
    print(f"A: {result['answer']}\n")
    print("   Sources: " + ", ".join(
        f"{d.metadata.get('source')} p{d.metadata.get('page')}" for d in result["context"]
    ) + "\n")

    conversation.append({"role": "user", "content": question})
    conversation.append({"role": "assistant", "content": result["answer"]})

## 10. Grounding check

The one property that separates RAG from a model guessing confidently: when the documents
do not contain the answer, it should say so rather than fill the gap from pretraining.

In [ ]:
OFF_TOPIC = "What is the current price of Bitcoin?"

result = bot.answer(OFF_TOPIC)
print(f"Q: {OFF_TOPIC}")
print(f"A: {result['answer']}")

## 11. Save the run

Written to [`outputs/`](./outputs) so a run can be inspected — or shown to someone —
without re-executing the notebook. Same pattern as Week 1.

In [ ]:
from datetime import datetime, timezone

record = {
    "saved_at": datetime.now(timezone.utc).isoformat(),
    "config": {
        "llm": config.GROQ_MODEL,
        "embedding_model": config.EMBEDDING_MODEL,
        "chunk_size": config.CHUNK_SIZE,
        "chunk_overlap": config.CHUNK_OVERLAP,
        "strategy": config.RETRIEVAL_STRATEGY,
        "top_k": config.TOP_K,
    },
    "corpus": {"files": list_sources(vectorstore), "chunks": count(vectorstore)},
    "conversation": conversation,
}

path = Path("outputs/rag_run.json")
path.parent.mkdir(exist_ok=True)
path.write_text(json.dumps(record, indent=2), encoding="utf-8")
print(f"Saved to {path}")

---

## What to try next

- Re-run section 5 with a question whose answer sits in one specific paragraph, and watch
  the strategies diverge more sharply than they do on a broad question.
- Set `CHUNK_SIZE=300` in `.env`, re-ingest, and see retrieval get more precise but the
  answers lose the surrounding context that made them readable.
- Switch `EMBEDDING_MODEL` to `sentence-transformers/all-MiniLM-L6-v2` with
  `EMBEDDING_BACKEND=huggingface` and compare retrieval quality on the same questions.
- Turn on `USE_RERANKER` and measure how much latency it costs you per question.

Then run the chatbot itself:

```
streamlit run app.py
```